# Bronze Layer: Holidays API CDC

**Source**: Nager.Date Public Holidays API (free, no auth required)  
**Target**: `workspace.bronze.holidays_api_cdc`  
**Primary Key**: **Composite** (`country_code`, `date`)

**Process:**
1. Fetch holidays from API for multiple countries/years
2. Convert to DataFrame with schema
3. Deduplicate on composite primary key
4. **MERGE**: Update existing, insert new (idempotent)

**Business Use Case:**
- Analyze holiday impact on sales patterns
- Identify sales spikes before major holidays
- Optimize inventory around holiday seasons

**Idempotent**: Re-run safe - no duplicates created

## Configuration

In [0]:
# Imports
import requests
import json
import time
from datetime import datetime
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType, BooleanType

# Target table
bronze_table = "workspace.bronze.holidays_api_cdc"
primary_keys = ["country_code", "date"]  # Composite key

# Country mapping for holidays API (ISO codes)
country_mapping = {
    "US": {"name": "United States", "iso_code": "US"},
    "GB": {"name": "United Kingdom", "iso_code": "GB"},
    "DE": {"name": "Germany", "iso_code": "DE"},
    "FR": {"name": "France", "iso_code": "FR"},
    "CA": {"name": "Canada", "iso_code": "CA"},
    "AU": {"name": "Australia", "iso_code": "AU"}
}

# Years to fetch (matching sales data range)
years = [2010, 2011, 2012, 2013, 2014]

print("✓ Configuration loaded")
print(f"  Target: {bronze_table}")
print(f"  Primary Key: {primary_keys}")
print(f"  Countries: {len(country_mapping)}")
print(f"  Years: {years[0]}-{years[-1]}")
print(f"  Total API calls: {len(country_mapping) * len(years)}")

## CDC Ingestion with MERGE

In [0]:
# Fetch public holidays from API
print("🌐 Fetching holidays from Nager.Date API...\n")

holidays_data = []

for country_code, country_info in country_mapping.items():
    print(f"🌍 {country_info['name']} ({country_code})")
    
    for year in years:
        print(f"  {year}...", end=" ")
        
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_info['iso_code']}"
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            holidays = response.json()
            
            # Add metadata to each holiday
            for holiday in holidays:
                holiday["country_code"] = country_code
                holiday["country_name"] = country_info["name"]
                holiday["year"] = year
                holiday["raw_json"] = json.dumps(holiday)  # Add raw JSON
                holidays_data.append(holiday)
            
            print(f"✓ {len(holidays)} holidays")
            time.sleep(0.5)  # Be polite to API
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"✓ Collected {len(holidays_data)} holidays from API")

# Convert to DataFrame
schema = StructType([
    StructField("country_code", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("year", StringType(), True),
    StructField("date", DateType(), True),
    StructField("holiday_name", StringType(), True),
    StructField("local_name", StringType(), True),
    StructField("is_global", BooleanType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

bronze_records = [
    (
        holiday["country_code"],
        holiday["country_name"],
        str(holiday["year"]),
        datetime.strptime(holiday["date"], "%Y-%m-%d").date(),
        holiday.get("name", ""),
        holiday.get("localName", ""),
        holiday.get("global", False),
        holiday.get("raw_json", ""),
        datetime.now()
    )
    for holiday in holidays_data
]

df_bronze_batch = spark.createDataFrame(bronze_records, schema)

# Deduplicate on primary keys before MERGE
df_bronze_batch = df_bronze_batch.dropDuplicates(primary_keys)

print(f"✓ Created DataFrame: {df_bronze_batch.count():,} unique records")
print("  Ready for MERGE operation")

In [0]:
# Write to Bronze table using MERGE
from delta.tables import DeltaTable

print("Starting CDC ingestion with MERGE logic...")

# Check if table exists
table_exists = spark.catalog.tableExists(bronze_table)

if not table_exists:
    # First run: Create table with initial data
    print("  Table doesn't exist - creating with initial load")
    (df_bronze_batch.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(bronze_table)
    )
    print(f"\n✓ Initial load complete: {spark.table(bronze_table).count():,} records")
else:
    # Subsequent runs: MERGE new/updated holidays
    print("  Table exists - running MERGE...")
    
    target_table = DeltaTable.forName(spark, bronze_table)
    
    # Build MERGE condition for composite key
    merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in primary_keys])
    
    # Perform MERGE operation
    (target_table.alias("target")
      .merge(
          df_bronze_batch.alias("source"),
          merge_condition
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute()
    )
    
    print(f"\n✓ MERGE complete: {spark.table(bronze_table).count():,} total records")

## Verification

Check ingestion results

In [0]:
%sql
-- Verify bronze table
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT country_code) as unique_countries,
  MIN(date) as earliest_date,
  MAX(date) as latest_date,
  MIN(ingestion_timestamp) as first_ingestion,
  MAX(ingestion_timestamp) as last_ingestion
FROM bronze.holidays_api_cdc

In [0]:
%sql
-- Sample records from bronze
SELECT *
FROM bronze.holidays_api_cdc
ORDER BY ingestion_timestamp DESC
LIMIT 10